<a href="https://colab.research.google.com/github/PrinzVike/CSC4110-G02/blob/main/notebooks_G02_U7_unsupervised_rl_ipynb_.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans, DBSCAN
from sklearn.metrics import silhouette_score
from sklearn.decomposition import PCA
from sklearn.neighbors import NearestNeighbors

from pathlib import Path

# Random seed
random_seed = 423

# Create output directory
Path("outputs").mkdir(exist_ok=True)

# Load dataset
df = pd.read_csv("insurance.csv")

print(df.head())
print(df.shape)

In [ ]:
# Select numerical features only
features = ['age', 'bmi', 'children', 'charges']

X = df[features]

# Scale features
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

print("Scaled features:")
print(X_scaled[:5])

In [ ]:
inertias = []
sil_scores = {}

for k in range(2, 11):

    kmeans = KMeans(
        n_clusters=k,
        random_state=423,
        n_init=10
    )

    labels = kmeans.fit_predict(X_scaled)

    inertias.append(kmeans.inertia_)

    sil_scores[k] = silhouette_score(
        X_scaled,
        labels
    )

print("Inertias:", inertias)
print("Silhouette Scores:", sil_scores)

In [ ]:
plt.figure(figsize=(8, 5))

plt.plot(range(2, 11), inertias, marker='o')

plt.xlabel("Number of Clusters (k)")
plt.ylabel("Inertia")
plt.title("K-Means Elbow Curve")

plt.grid(True)
plt.savefig("outputs/elbow.png", dpi=150)
plt.show()

In [ ]:
best_k = max(sil_scores, key=sil_scores.get)

print("Best K:", best_k)
print("Best Silhouette Score:", sil_scores[best_k])

In [ ]:
kmeans = KMeans(
    n_clusters=best_k,
    random_state=423,
    n_init=10
)

clusters = kmeans.fit_predict(X_scaled)

df['cluster'] = clusters

cluster_means = df.groupby('cluster')[features].mean()

print("Mean values per cluster:")
display(cluster_means)

In [ ]:
pca = PCA(n_components=2)

X_pca = pca.fit_transform(X_scaled)

print("Explained Variance Ratio:")
print(pca.explained_variance_ratio_)

print("Total Explained Variance:")
print(sum(pca.explained_variance_ratio_))

In [ ]:
plt.figure(figsize=(8, 6))

plt.scatter(
    X_pca[:, 0],
    X_pca[:, 1],
    c=clusters,
    cmap='viridis',
    alpha=0.7
)

plt.xlabel("Principal Component 1")
plt.ylabel("Principal Component 2")
plt.title("PCA Visualization of K-Means Clusters")

plt.colorbar(label="Cluster")

plt.savefig("outputs/pca_clusters.png", dpi=150)
plt.show()

In [ ]:
neighbors = NearestNeighbors(n_neighbors=5)

neighbors_fit = neighbors.fit(X_scaled)

distances, indices = neighbors_fit.kneighbors(X_scaled)

k_distances = np.sort(distances[:, -1])

plt.figure(figsize=(8, 5))

plt.plot(k_distances)

plt.xlabel("Data Points")
plt.ylabel("5th Nearest Neighbor Distance")
plt.title("DBSCAN K-Distance Plot")

plt.grid(True)
plt.show()

In [ ]:
eps = float(np.percentile(k_distances, 90))

print("Initial eps:", eps)

In [ ]:
dbscan = DBSCAN(
    eps=eps,
    min_samples=5
)

dbscan_labels = dbscan.fit_predict(X_scaled)

df['dbscan_cluster'] = dbscan_labels

print("DBSCAN labels:")
print(np.unique(dbscan_labels))

In [ ]:
# K-Means statistics
kmeans_clusters = len(np.unique(clusters))
kmeans_noise = 0

# DBSCAN statistics
dbscan_clusters = len(set(dbscan_labels) - {-1})
dbscan_noise = np.sum(dbscan_labels == -1)

print("K-Means clusters:", kmeans_clusters)
print("K-Means noise points:", kmeans_noise)

print("DBSCAN clusters:", dbscan_clusters)
print("DBSCAN noise points:", dbscan_noise)

In [ ]:
print("Smoker distribution in K-Means clusters:")

print(
    pd.crosstab(
        df['cluster'],
        df['smoker'],
        normalize='index'
    ).round(3)
)

print("\nSmoker distribution in DBSCAN clusters:")

print(
    pd.crosstab(
        df['dbscan_cluster'],
        df['smoker'],
        normalize='index'
    ).round(3)
)

In [ ]:
!pip install gymnasium[toy-text]
import gymnasium as gym
import numpy as np
import matplotlib.pyplot as plt

env = gym.make(
    "FrozenLake-v1",
    is_slippery=False
)

print("States:", env.observation_space.n)
print("Actions:", env.action_space.n)

In [ ]:
Q = np.zeros((16, 4))

alpha = 0.1
gamma = 0.99
epsilon = 1.0

epsilon_min = 0.05
epsilon_decay = 0.995

episodes = 2000

success_rates = []

In [ ]:
env = gym.make(
    "FrozenLake-v1",
    is_slippery=False
)

Q = np.zeros((16, 4))

success_history = []

for episode in range(episodes):

    state, info = env.reset(seed=423 + episode)

    done = False
    successful = 0

    epsilon = max(
        epsilon_min,
        1.0 * (epsilon_decay ** episode)
    )

    while not done:

        # Epsilon-greedy action selection
        if np.random.random() < epsilon:

            action = env.action_space.sample()

        else:

            action = np.argmax(Q[state])

        next_state, reward, terminated, truncated, info = env.step(action)

        done = terminated or truncated

        # Q-learning update
        old_value = Q[state, action]

        next_max = np.max(Q[next_state])

        if terminated:
            next_max = 0

        new_value = old_value + alpha * (
            reward + gamma * next_max - old_value
        )

        Q[state, action] = new_value

        state = next_state

        if done and reward == 1:
            successful = 1

    success_history.append(successful)

    # Store success rate every 100 episodes
    if (episode + 1) % 100 == 0:

        success_rate = np.mean(success_history[-100:])

        success_rates.append(success_rate)

print("Training completed!")
print("Success rates:", success_rates)

env.close()

In [ ]:
plt.figure(figsize=(8, 5))

plt.plot(
    range(100, 2001, 100),
    success_rates,
    marker='o'
)

plt.xlabel("Episodes")
plt.ylabel("Success Rate")

plt.title("Q-Learning Success Rate")

plt.grid(True)

plt.savefig("outputs/success_rate.png", dpi=150)

plt.show()

In [ ]:
arrows = {
    0: '←',
    1: '↓',
    2: '→',
    3: '↑'
}

policy = np.argmax(Q, axis=1)

lake = [
    'SFFF',
    'FHFH',
    'FFFH',
    'HFFG'
]

for i in range(4):

    row = []

    for j in range(4):

        state = i * 4 + j

        cell = lake[i][j]

        if cell == 'H' or cell == 'G':

            row.append(cell)

        else:

            row.append(arrows[policy[state]])

    print(" ".join(row))

In [ ]:
if episode == 0 and state == 0 and 'update_example' not in globals():

    update_example = {
        "Q_before": old_value,
        "reward": reward,
        "best_next_Q": next_max,
        "Q_after": new_value
    }

In [ ]:
print("Q-learning update example:")

for key, value in update_example.items():

    print(key, ":", value)

In [ ]:
env = gym.make(
    "FrozenLake-v1",
    is_slippery=True
)

Q_slippery = np.zeros((16, 4))

success_slippery = []

for episode in range(2000):

    state, info = env.reset(seed=423 + episode)

    done = False
    successful = 0

    epsilon = max(
        0.05,
        1.0 * (0.995 ** episode)
    )

    while not done:

        if np.random.random() < epsilon:

            action = env.action_space.sample()

        else:

            action = np.argmax(Q_slippery[state])

        next_state, reward, terminated, truncated, info = env.step(action)

        done = terminated or truncated

        old_value = Q_slippery[state, action]

        next_max = np.max(Q_slippery[next_state])

        if terminated:
            next_max = 0

        new_value = old_value + 0.1 * (
            reward + 0.99 * next_max - old_value
        )

        Q_slippery[state, action] = new_value

        state = next_state

        if done and reward == 1:
            successful = 1

    success_slippery.append(successful)

env.close()

print("Slippery training completed!")
print("Final 100-episode success rate:",
      np.mean(success_slippery[-100:]))